[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Guía del tema 01](README.md)

# Jerarquía de memoria y Roofline

**Tema:** 01 · **Sesiones:** 5, 6 · **Edición:** 1.0.2026

**Pregunta guía:** ¿La ejecución está limitada por cómputo, ancho de banda, latencia o localidad?


## Cómo usar este notebook

Sigue la secuencia sin saltar directamente al código:

1. Comprueba los prerrequisitos y formula una respuesta inicial a la pregunta guía.
2. Estudia la explicación paso a paso y reconstruye el mapa visual.
3. Predice el resultado del ejemplo resuelto antes de ejecutar su celda.
4. Repite el razonamiento en el ejemplo guiado y contesta las preguntas de comprensión.
5. Solo entonces desarrolla los ejercicios progresivos y contrasta los criterios de aceptación.

La meta no es memorizar una salida: es poder explicar qué se calculó, bajo qué supuestos y con qué evidencia.


## Antes de empezar

**Por qué importa.** El procesador solo calcula con datos que logró mover hasta una unidad de ejecución. Roofline organiza esa tensión entre cómputo, ancho de banda y reutilización.

**Prerrequisitos.**

- Aritmética, funciones y lectura de gráficas.
- Python básico para modificar parámetros y ejecutar aserciones.

**Diagnóstico inicial.** Escribe una respuesta de dos frases a la pregunta guía. Al terminar, vuelve a leerla y señala qué corregiste.


## Resultados de aprendizaje

Al finalizar podrás:

- Calcular intensidad aritmética.
- Aplicar el límite Roofline sin confundirlo con una medición.
- Reconocer localidad, NUMA y false sharing como causas observables.


## Explicación paso a paso

En esta sección todavía no se busca programar. Primero se construye el modelo mental que permitirá leer el ejemplo y detectar conclusiones inválidas.

### Paso 1: construye la idea

Roofline acota rendimiento por min(pico, ancho de banda × intensidad).

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Paso 2: construye la idea

Una línea de caché compartida por escrituras de varios núcleos puede invalidarse aunque las variables sean distintas.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Paso 3: construye la idea

En NUMA, ubicación de memoria, first-touch y afinidad forman parte del experimento.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Vocabulario mínimo

- trabajo T₁ — costo total de todas las operaciones
- span T∞ — costo del camino dependiente más largo
- eficiencia — aceleración dividida entre recursos
- intensidad aritmética — FLOP realizados por byte movido


## Mapa visual

Los diagramas se almacenan en la carpeta compartida [`curso/images/`](../../images/README.md). Úsalos para explicar relaciones y secuencias; no los trates como resultados experimentales.

### Jerarquia Memoria

![Jerarquía de memoria y costo de movimiento](../../images/jerarquia-memoria.svg)

**Cómo leerlo.** Al descender aumenta la capacidad y suele aumentar la latencia. La optimización busca reutilizar datos antes de solicitar un nivel más lejano.

### Metodo Rendimiento

![Ciclo de medición, resumen, perfil e hipótesis](../../images/metodo-rendimiento.svg)

**Cómo leerlo.** Una medición se repite y resume antes de perfilar. La conclusión genera un experimento nuevo cambiando una sola variable controlada.


In [ ]:
from pathlib import Path

def find_repository(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "INDICE_CURSO.md").is_file():
            return candidate
    raise RuntimeError("No se encontró la raíz del repositorio")

ROOT = find_repository(Path.cwd())
TOPIC = "01"
NOTEBOOK = "01_fundamentos/03_memoria_roofline.ipynb"
assert (ROOT / "curso" / "notebooks" / "01_fundamentos" / "README.md").is_file()
print(f"Repositorio: {ROOT}")
print(f"Notebook: {NOTEBOOK}")


## Ejemplo resuelto: Límite Roofline

**Situación.** Se calcula el techo para varias intensidades con unidades coherentes.

**Razonamiento antes del código.**

1. Identifica entradas, supuestos y la magnitud que debe producirse.
2. Formula una propiedad esperada; la celda la expresa mediante una aserción.
3. Predice el resultado y después ejecuta. Una salida impresa ayuda a observar, pero la aserción decide si se conserva el invariante.


In [ ]:
peak_gflops = 800.0
bandwidth_gbs = 120.0
intensities = (0.125, 0.5, 1, 2, 4, 8, 16)
ridge = peak_gflops / bandwidth_gbs
assert abs(ridge - 20/3) < 1e-12
print(f"punto de transición = {ridge:.2f} FLOP/byte")
for intensity in intensities:
    ceiling = min(peak_gflops, bandwidth_gbs * intensity)
    assert 0 < ceiling <= peak_gflops
    regime = "memoria" if bandwidth_gbs * intensity < peak_gflops else "cómputo"
    print(f"I={intensity:6.3f} techo={ceiling:7.1f} GFLOP/s régimen={regime}")


### Explicación del resultado

El techo no es rendimiento obtenido: se compara con mediciones de la misma precisión y operación.

**Qué debes poder explicar.** Relaciona cada valor producido con el modelo conceptual y distingue el cálculo ilustrativo de una medición sobre hardware real.


## Ejemplo guiado: Líneas de caché

**Situación.** Se observa cuándo contadores adyacentes comparten una línea de 64 bytes.

**Tu turno antes de ejecutar.** Anota una predicción, identifica la variable que modificarías y explica qué propiedad no debe cambiar. Ejecuta después y compara el resultado con tu predicción.


In [ ]:
line_size = 64
element_size = 8
addresses = [i * element_size for i in range(16)]
mapping = {i: address // line_size for i, address in enumerate(addresses)}
assert len({mapping[i] for i in range(8)}) == 1
for index, line in mapping.items(): print(f"contador {index:2} -> línea {line}")
print("separación mínima en elementos:", line_size // element_size)


### Lectura razonada

Separar o alinear contadores puede reducir false sharing, pero aumenta memoria y debe medirse.

**Qué debes poder explicar.** Relaciona cada valor producido con el modelo conceptual y distingue el cálculo ilustrativo de una medición sobre hardware real.


## Comprueba tu comprensión

1. ¿Qué medición permitiría distinguir baja intensidad aritmética de mala localidad o de un pico teórico inadecuado?
2. ¿Qué aserción o comparación del ejemplo protege la corrección y qué error detectaría?
3. ¿Qué parte es un modelo y qué evidencia adicional exigirías antes de generalizar al hardware real?

Responde primero sin ejecutar código. Luego usa las celdas anteriores para corregir o precisar tu explicación.


## Ejercicios progresivos

### Nivel 1 — reproducir y explicar

Cambia un parámetro del ejemplo resuelto, predice el efecto y explica por qué la aserción debe seguir pasando o debe fallar de manera controlada.

### Nivel 2 — aplicar

1. Estimar bytes transferidos y FLOP de un kernel.
2. Medir una referencia con tamaño que exceda caché cuando la pregunta sea ancho de banda.
3. Registrar afinidad y política NUMA junto con la curva.

### Nivel 3 — producir evidencia

Conserva entrada, comandos, versión del entorno, resultados crudos y una conclusión limitada por los supuestos. Separa siempre corrección, tiempo de kernel y tiempo extremo a extremo cuando corresponda.

### Actividades compilables relacionadas

- [Ejercicio C17: vector triad y Roofline](../../ejercicios/01_fundamentos/02_ancho_banda_roofline/README.md)


## Errores frecuentes

- Usar FLOP/s de pico de otra precisión.
- Confundir misses con prueba automática de false sharing.
- Comparar tamaños que realizan cantidades distintas de trabajo.


## Criterios de aceptación

- Unidades y precisión explícitas.
- Punto Roofline calculado y medición diferenciada.
- Hipótesis de memoria contrastada con al menos un contador o experimento controlado.


## Síntesis

- La pregunta que debes poder responder es: **¿La ejecución está limitada por cómputo, ancho de banda, latencia o localidad?**
- Los ejemplos convierten el modelo en propiedades comprobables; no sustituyen una medición del sistema objetivo.
- Los ejercicios se consideran terminados cuando la explicación, la corrección y la evidencia satisfacen los criterios de aceptación.


## Referencias y material relacionado

- [Planeación: memoria](../../../docs/PLANEACION_CURSO.md)
- [Protocolo](../../../docs/REPRODUCIBILIDAD_EJERCICIOS.md)


[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Continuar desde la guía del tema 01](README.md)
